# Environmental vulnerabilities (power, HVAC, fire, water)

# Coding problem, 2.2.b: Environmental sensor triage

**Skill:** Analyze Risk · **Maps to:** 2.2.A · **Language:** Python 3 (pytest)

## Problem
A data center streams readings from environmental sensors placed around each server room:
power (voltage as a percent of nominal), temperature (Celsius), humidity (percent relative
humidity), smoke detectors, and water/flood detectors. Facilities staff need every reading
automatically triaged into a severity level so a human only has to look at what matters.

Your task is to write a function `solve(data)` that takes a list of sensor readings and returns
a dictionary mapping each reading's `id` to a severity string: `"NORMAL"`, `"WARNING"`, or
`"CRITICAL"`.

### Per-reading thresholds (by `type`)

**`power`** (`value` = supply voltage as a percent of nominal, 100 = nominal):
- `CRITICAL` if `value == 0` (total outage) or `value < 90` or `value > 110`
- `WARNING` if `90 <= value < 95` or `105 < value <= 110`
- `NORMAL` otherwise (`95 <= value <= 105`)

**`temperature`** (`value` = degrees Celsius):
- `CRITICAL` if `value > 30`
- `WARNING` if `27 < value <= 30` or `value < 15`
- `NORMAL` otherwise (`15 <= value <= 27`)

**`humidity`** (`value` = percent relative humidity):
- `CRITICAL` if `value > 70` or `value < 20`
- `WARNING` if `60 < value <= 70` or `20 <= value < 40`
- `NORMAL` otherwise (`40 <= value <= 60`)

**`smoke`** (`value` = detector reading, `0` = clear, any positive value = smoke detected):
- `CRITICAL` if `value > 0`
- `NORMAL` otherwise

**`water`** (`value` = detector reading, `0` = dry, any positive value = moisture detected):
- `CRITICAL` if `value > 0`
- `NORMAL` otherwise

### Combined-signal escalation (cooling-failure pattern)
After computing each reading's per-reading severity above, apply one escalation rule: for
readings that share the same `room`, if there is **both** a `temperature` reading rated
`WARNING` or `CRITICAL` **and** a `humidity` reading rated `WARNING` or `CRITICAL` in that same
room, escalate *both* of those specific readings (temperature and humidity only) to
`CRITICAL`. This models a failing HVAC unit, where rising temperature and drifting humidity
together are a stronger signal than either alone. Other reading types in the room (power, smoke,
water) are not affected by this rule, and a room where only one of the two (temperature or
humidity) is elevated is not escalated.

## Function contract
Implement `solve(data: list[dict]) -> dict[str, str]` in the code editor.

## Input
A list of dictionaries, each representing one sensor reading:
```python
[
    {"id": "s1", "room": "server-room-a", "type": "temperature", "value": 29},
    {"id": "s2", "room": "server-room-a", "type": "humidity", "value": 65},
    {"id": "s3", "room": "server-room-b", "type": "power", "value": 100},
]
```

## Output
A dictionary mapping each reading's `"id"` to its severity string:
```python
{
    "s1": "CRITICAL",   # escalated: room also had a humidity WARNING/CRITICAL
    "s2": "CRITICAL",   # escalated: room also had a temperature WARNING/CRITICAL
    "s3": "NORMAL",
}
```

## Constraints
- `type` is one of `"power"`, `"temperature"`, `"humidity"`, `"smoke"`, `"water"`.
- `value` is a non-negative number (float or int) except `power`, which may be `0` (outage).
- Pure function, deterministic, standard library only.

## How to run
```bash
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(data: list[dict]) -> dict[str, str]
```

### Input (`data`)
A list of dictionaries representing sensor readings. Each dictionary contains:
- `id` (str): Unique identifier for the reading.
- `room` (str): Room/zone identifier the sensor is located in.
- `type` (str): One of `"power"`, `"temperature"`, `"humidity"`, `"smoke"`, `"water"`.
- `value` (float/int): The sensor's numeric reading (see the problem statement above for units per type).

### Output (`result`)
A dictionary where keys are the `id` of each reading from the input and values are the severity
string: `"NORMAL"`, `"WARNING"`, or `"CRITICAL"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- Per-type thresholds (see the problem statement above for full detail):
  1. `power`: outage (`0`) or `<90` or `>110` -> `CRITICAL`; `[90,95)` or `(105,110]` -> `WARNING`; else `NORMAL`.
  2. `temperature`: `>30` -> `CRITICAL`; `(27,30]` or `<15` -> `WARNING`; else `NORMAL`.
  3. `humidity`: `>70` or `<20` -> `CRITICAL`; `(60,70]` or `[20,40)` -> `WARNING`; else `NORMAL`.
  4. `smoke`/`water`: any positive value -> `CRITICAL`; `0` -> `NORMAL`.
  5. Combined-signal escalation: within the same `room`, if a `temperature` reading is
     `WARNING`/`CRITICAL` AND a `humidity` reading is `WARNING`/`CRITICAL`, escalate both of
     those two readings to `CRITICAL` (only temperature and humidity readings are affected).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 2.2.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import Any


def solve(data: list[dict]) -> dict[str, str]:
    """Triage a list of environmental sensor readings into severity levels.

    Args:
        data: A list of dicts, each with keys id, room, type, value.
    Returns:
        A dictionary mapping reading ids to "NORMAL" | "WARNING" | "CRITICAL".
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_power_levels():
    data = [
        {"id": "p1", "room": "a", "type": "power", "value": 0},
        {"id": "p2", "room": "a", "type": "power", "value": 92},
        {"id": "p3", "room": "a", "type": "power", "value": 100},
        {"id": "p4", "room": "a", "type": "power", "value": 108},
        {"id": "p5", "room": "a", "type": "power", "value": 120},
    ]
    assert impl.solve(data) == {
        "p1": "CRITICAL",
        "p2": "WARNING",
        "p3": "NORMAL",
        "p4": "WARNING",
        "p5": "CRITICAL",
    }


def test_temperature_and_humidity_no_escalation():
    # Only temperature elevated in room b, humidity stays NORMAL -> no escalation.
    data = [
        {"id": "t1", "room": "b", "type": "temperature", "value": 28},
        {"id": "h1", "room": "b", "type": "humidity", "value": 50},
    ]
    assert impl.solve(data) == {"t1": "WARNING", "h1": "NORMAL"}


def test_smoke_and_water():
    data = [
        {"id": "s1", "room": "c", "type": "smoke", "value": 0},
        {"id": "s2", "room": "c", "type": "smoke", "value": 3},
        {"id": "w1", "room": "c", "type": "water", "value": 0},
        {"id": "w2", "room": "c", "type": "water", "value": 1},
    ]
    assert impl.solve(data) == {
        "s1": "NORMAL",
        "s2": "CRITICAL",
        "w1": "NORMAL",
        "w2": "CRITICAL",
    }


def test_combined_escalation():
    # Room d: temperature WARNING + humidity WARNING -> both escalate to CRITICAL.
    # Power reading in same room is unaffected.
    data = [
        {"id": "t1", "room": "d", "type": "temperature", "value": 29},
        {"id": "h1", "room": "d", "type": "humidity", "value": 65},
        {"id": "p1", "room": "d", "type": "power", "value": 100},
    ]
    assert impl.solve(data) == {"t1": "CRITICAL", "h1": "CRITICAL", "p1": "NORMAL"}


def test_accept():
    data = [
        {"id": "t1", "room": "e", "type": "temperature", "value": 20},
        {"id": "h1", "room": "e", "type": "humidity", "value": 45},
    ]
    assert impl.solve(data) == {"t1": "NORMAL", "h1": "NORMAL"}


# ---- hidden / edge cases -------------------------------------------------
def test_edge_empty():
    assert impl.solve([]) == {}


def test_multiple_rooms_independent():
    data = [
        {"id": "t1", "room": "room1", "type": "temperature", "value": 31},
        {"id": "h1", "room": "room2", "type": "humidity", "value": 75},
        {"id": "t2", "room": "room2", "type": "temperature", "value": 20},
    ]
    # room1: only temperature elevated (already CRITICAL, no escalation needed to check)
    # room2: humidity CRITICAL but temperature NORMAL -> no escalation of temperature
    assert impl.solve(data) == {"t1": "CRITICAL", "h1": "CRITICAL", "t2": "NORMAL"}


def test_already_critical_stays_critical_after_escalation():
    data = [
        {"id": "t1", "room": "f", "type": "temperature", "value": 35},
        {"id": "h1", "room": "f", "type": "humidity", "value": 65},
    ]
    assert impl.solve(data) == {"t1": "CRITICAL", "h1": "CRITICAL"}


def test_low_temperature_and_low_humidity_escalate():
    data = [
        {"id": "t1", "room": "g", "type": "temperature", "value": 10},
        {"id": "h1", "room": "g", "type": "humidity", "value": 25},
    ]
    assert impl.solve(data) == {"t1": "CRITICAL", "h1": "CRITICAL"}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
